In [ ]:
%load_ext autoreload
%autoreload 2

# 1. DQN Agent

In [ ]:
import numpy as np
import torch

from briscola import Briscola
from agents.dqn_agent import DQN_Agent

In [ ]:
MODEL = "models/150000_dqn.pt"

device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)

In [ ]:
N_EPISODES = 10_000

env   = Briscola()
save = torch.load(MODEL, map_location = device)
agent = DQN_Agent(
    env=env,
    device=device,
    savefile=save,
    epsilon_start=0.0, # no exploration during eval
    epsilon_end=0.0,
)
agent.policy_net.eval()

# Trackers
results = [] # "win" / "loss" / "tie"
score_diffs = [] # player_score - opponent_score
episode_rewards = []
points_in_outcome = {"win": [], "loss": [], "tie": []}

for ep in range(N_EPISODES):
    state, info = env.reset()
    ep_reward = 0.0
    done = False

    while not done:
        action = agent.select_action(state).item()
        state, reward, terminated, truncated, info = env.step(action)
        ep_reward += reward
        done = terminated or truncated

    episode_rewards.append(ep_reward)

    if env.player_score > 6:
        outcome = "win"
    elif env.player_score < 6:
        outcome = "loss"
    else:
        outcome = "tie"

    results.append(outcome)
    points_in_outcome[outcome].append(env.player_score)

# Report
wins = results.count("win")
losses = results.count("loss")
ties = results.count("tie")

print(f"\n{'='*52}")
print(f"  Evaluation over {N_EPISODES} episodes")
print(f"{'='*52}")
print(f"  Win rate:   {wins/N_EPISODES*100:5.1f}%  ({wins})")
print(f"  Loss rate:  {losses/N_EPISODES*100:5.1f}%  ({losses})")
print(f"  Tie rate:   {ties/N_EPISODES*100:5.1f}%  ({ties})")
print(f"{'─'*52}")
print(f"  Avg shaped reward per episode:      {np.mean(episode_rewards):+.2f}")
print(f"{'─'*52}")
for outcome in ("win", "loss", "tie"):
    pts = points_in_outcome[outcome]
    if pts:
        print(f"  Avg agent points in {outcome}s:         {np.mean(pts):.1f}")
print(f"{'='*52}\n")

# 2. PPO Agent

In [ ]:
import numpy as np
import torch

from briscola import Briscola
from agents.ppo_agent import PPO_Agent

In [ ]:
MODEL = "models/1000000_ppo.pt"

device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)

In [ ]:
N_EPISODES = 10_000

env   = Briscola()
save = torch.load(MODEL, map_location = device)
agent = PPO_Agent(
    env=env,
    device=device,
    savefile=save,
)
agent.policy_net.eval()

# Trackers
results = [] # "win" / "loss" / "tie"
score_diffs = [] # player_score - opponent_score
episode_rewards = []
points_in_outcome = {"win": [], "loss": [], "tie": []}

for ep in range(N_EPISODES):
    state, info = env.reset()
    ep_reward = 0.0
    done = False

    while not done:
        action, mask, log_prob, value = agent.select_action(state)
        state, reward, terminated, truncated, info = env.step(action)
        ep_reward += reward
        done = terminated or truncated

    episode_rewards.append(ep_reward)

    if env.player_score > 6:
        outcome = "win"
    elif env.player_score < 6:
        outcome = "loss"
    else:
        outcome = "tie"

    results.append(outcome)
    points_in_outcome[outcome].append(env.player_score)

# Report
wins = results.count("win")
losses = results.count("loss")
ties = results.count("tie")

print(f"\n{'='*52}")
print(f"  Evaluation over {N_EPISODES} episodes")
print(f"{'='*52}")
print(f"  Win rate:   {wins/N_EPISODES*100:5.1f}%  ({wins})")
print(f"  Loss rate:  {losses/N_EPISODES*100:5.1f}%  ({losses})")
print(f"  Tie rate:   {ties/N_EPISODES*100:5.1f}%  ({ties})")
print(f"{'─'*52}")
print(f"  Avg shaped reward per episode:      {np.mean(episode_rewards):+.2f}")
print(f"{'─'*52}")
for outcome in ("win", "loss", "tie"):
    pts = points_in_outcome[outcome]
    if pts:
        print(f"  Avg agent points in {outcome}s:         {np.mean(pts):.1f}")
print(f"{'='*52}\n")